# Music Energy Classifier
Fine-tuning ModernBERT to classify text descriptions of music into energy levels 0–5.

| Label | Vibe |
|---|---|
| 0 | Chill / ambient / sleep |
| 1 | Mellow / acoustic / lo-fi |
| 2 | Mid-tempo / background |
| 3 | Upbeat / feel-good / danceable |
| 4 | High energy / hype / workout |
| 5 | Absolute banger / chaos |

## 1. Imports

In [2]:
import json
import time
import warnings
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

PyTorch version: 2.10.0
MPS available: True


## 2. Load Training Data
Load the labeled JSON examples and split 80% for training, 20% for testing.

In [3]:
with open("./data/feedback-classifier.json", "r") as f:
    data = json.load(f)

# Convert to Hugging Face Dataset
dataset = Dataset.from_list(data)

# Split 80/20 train/test
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Training samples:   {len(dataset['train'])}")
print(f"Test samples:       {len(dataset['test'])}")
print(f"\nSample entry:")
print(dataset['train'][0])

Training samples:   62
Test samples:       16

Sample entry:
{'text': 'Laid-back funk with a pocket bass line and light percussion', 'label': 2}


## 3. Tokenization
Convert text into token IDs that ModernBERT understands. Each word (or subword) maps to a unique integer.

In [4]:
model_id = "answerdotai/ModernBERT-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Show what tokenization looks like for one example
sample_text = dataset['train'][0]['text']
tokens = tokenizer(sample_text)
print(f"Text:   {sample_text}")
print(f"Tokens: {tokens['input_ids']}")
print(f"Decoded: {tokenizer.convert_ids_to_tokens(tokens['input_ids'])}")

Map: 100%|██████████| 16/16 [00:00<00:00, 7342.33 examples/s]

Text:   Laid-back funk with a pocket bass line and light percussion
Tokens: [50281, 7647, 301, 14, 2135, 794, 76, 342, 247, 11320, 16819, 1386, 285, 1708, 49298, 50282]
Decoded: ['[CLS]', 'La', 'id', '-', 'back', 'Ġfun', 'k', 'Ġwith', 'Ġa', 'Ġpocket', 'Ġbass', 'Ġline', 'Ġand', 'Ġlight', 'Ġpercussion', '[SEP]']


## 4. Model Setup
Load ModernBERT with a classification head for 6 energy labels.
The pre-trained weights already understand language — we're just teaching it our specific task.

In [5]:
id2label = {
    0: "Energy_0",
    1: "Energy_1",
    2: "Energy_2",
    3: "Energy_3",
    4: "Energy_4",
    5: "Energy_5"
}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {model_id}")
print(f"Total parameters: {total_params:,}")

Loading weights: 100%|██████████| 172/172 [00:00<00:00, 6518.90it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: answerdotai/ModernBERT-large
Total parameters: 395,837,446


## 5. Metrics
After each evaluation pass, compute accuracy, F1, precision, and recall.

In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("compute_metrics function defined")

compute_metrics function defined


## 6. Training Configuration
Speed mode: 3 epochs, fast iteration. Flip `USE_SPEED_MODE = False` for a full 10-epoch run optimized for F1.

In [7]:
USE_SPEED_MODE = True  # Set to False for performance mode

training_args_speed = TrainingArguments(
    output_dir="../models/music-energy-classifier/training-output",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none",
    fp16=False,
    warmup_ratio=0.1,
    logging_steps=10,
)

training_args_performance = TrainingArguments(
    output_dir="../models/music-energy-classifier/training-output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    report_to="none",
    fp16=False,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_total_limit=3,
)

training_args = training_args_speed if USE_SPEED_MODE else training_args_performance

print(f"Mode: {'SPEED' if USE_SPEED_MODE else 'PERFORMANCE'}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Mode: SPEED
Epochs: 3
Learning rate: 5e-05
Batch size: 16


## 7. Train
This is where the magic happens. The Trainer handles batching, forward/backward passes, gradient updates, and checkpointing.

> **Note:** ModernBERT-large is ~400MB. First run will download it. Training on Apple Silicon MPS takes a few minutes in speed mode.

In [8]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start_time = time.time()
train_result = trainer.train()
training_duration = time.time() - start_time

print(f"\nTraining complete in {training_duration:.1f}s")
print(f"Final training loss: {train_result.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/Users/mjohnson/dev/dev-gym-slm/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,2.077602,0.000000,0.000000,0.000000,0.000000
2,No log,1.461711,0.437500,0.401786,0.416667,0.437500
3,1.371600,1.502284,0.375000,0.338095,0.383333,0.375000


/Users/mjohnson/dev/dev-gym-slm/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mjohnson/dev/dev-gym-slm/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]
/Users/mjohnson/dev/dev-gym-slm/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader


Training complete in 21.1s
Final training loss: 1.2201


## 8. Evaluate
Run the model against the held-out test set and see how it performs.

In [9]:
# Pull final eval results from training log (last eval entry)
eval_logs = [x for x in trainer.state.log_history if "eval_accuracy" in x]
eval_results = eval_logs[-1]

print("=" * 50)
print("MODEL PERFORMANCE")
print("=" * 50)
print(f"Accuracy:  {eval_results['eval_accuracy']:.4f}  ({eval_results['eval_accuracy']*100:.1f}%)")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"Loss:      {eval_results['eval_loss']:.4f}")
print()

acc = eval_results['eval_accuracy']
if acc < 0.6:
    print("Low accuracy — add more training data")
elif acc < 0.8:
    print("Moderate accuracy — consider more examples or epochs")
elif acc < 0.9:
    print("Good accuracy")
else:
    print("Excellent accuracy")

MODEL PERFORMANCE
Accuracy:  0.3750  (37.5%)
F1 Score:  0.3381
Precision: 0.3833
Recall:    0.3750
Loss:      1.5023

Low accuracy — add more training data


## 9. Save the Model
Save the trained model and tokenizer to disk. The inference API will load from this directory.

In [10]:
save_path = "../models/music-energy-classifier/final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]

Model saved to ../models/music-energy-classifier/final
